# Session 2: G-code, offsets, arcs and time
**A. Real-world problem.** A CNC controller reads text lines such as `G1 X10 Y5 F300`.
Can you write that text yourself, read it back, and predict how long a 40 x 30 mm pocket,
5 mm deep, takes with a 6 mm cutter?

**Plan.** (1) write G0/G1 lines, (2) parse them back, (3) offset an outline by the cutter
radius, (4) add arcs G2/G3, (5) plan a pocket and estimate its time.

In [ ]:
import math
import pathlib
import re

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: write G-code words
A G-code line is a list of *words*: a letter and a number. `G0` = rapid move (fast, not
cutting), `G1` = straight cutting move, `X Y Z` = target coordinates in mm, `F` = feed in
mm/min. We assume millimetres (G21) and absolute coordinates (G90).

In [ ]:
def num(v):
    """Format a number for G-code: at most 3 decimals, no trailing zeros."""
    s = f"{v:.3f}".rstrip("0").rstrip(".")
    return "0" if s in ("-0", "") else s


class Program:
    """A tiny G-code writer. The text lives in self.lines."""

    def __init__(self):
        self.lines = ["G21 (millimetres)", "G90 (absolute coordinates)"]

    def rapid(self, x=None, y=None, z=None):
        self.lines.append("G0" + self._xyz(x, y, z))

    def cut(self, x=None, y=None, z=None, f=None):
        self.lines.append("G1" + self._xyz(x, y, z) + (f" F{num(f)}" if f is not None else ""))

    def arc(self, ccw, x, y, i, j, f=None):
        """G3 (counter-clockwise) or G2 (clockwise) arc to (x, y); (i, j) = centre minus start."""
        self.lines.append(("G3" if ccw else "G2") + f" X{num(x)} Y{num(y)} I{num(i)} J{num(j)}"
                          + (f" F{num(f)}" if f is not None else ""))

    @staticmethod
    def _xyz(x, y, z):
        return "".join(f" {k}{num(v)}" for k, v in (("X", x), ("Y", y), ("Z", z)) if v is not None)

    def text(self):
        return "\n".join(self.lines) + "\n"

## Step 2: parse your own G-code back
Reading the text back and rebuilding the geometry is how you *verify* a program. Words are
modal: a missing X, Y, Z keeps its last value, and G and F stay active until changed.

In [ ]:
WORD = re.compile(r"([A-Z])\s*(-?\d+\.?\d*|-?\.\d+)")


def parse_gcode(text):
    """Return a list of moves {kind, p0, p1, f, ij}. kind is G0, G1, G2 or G3."""
    pos, kind, feed, moves = [0.0, 0.0, 0.0], None, None, []
    for raw in text.splitlines():
        line = re.sub(r"\(.*?\)|;.*", "", raw).upper()
        words = {m.group(1): float(m.group(2)) for m in WORD.finditer(line)}
        g = words.get("G")
        if g in (0, 1, 2, 3):
            kind = f"G{int(g)}"
        if "F" in words:
            feed = words["F"]
        if not any(k in words for k in "XYZ"):
            continue
        new = [words.get("X", pos[0]), words.get("Y", pos[1]), words.get("Z", pos[2])]
        moves.append({"kind": kind, "p0": tuple(pos), "p1": tuple(new), "f": feed,
                      "ij": (words.get("I", 0.0), words.get("J", 0.0))})
        pos = new
    return moves

## Step 3: lengths, arcs and time
A line has length |p1 - p0|. An arc with centre c has radius r = |p0 - c| and, if it is
correct, also |p1 - c| = r. Its length is r times the swept angle (radians).
Machining time = length / feed (mm divided by mm/min gives minutes).

In [ ]:
def arc_geometry(move):
    """(centre, radius, sweep in radians) of a G2/G3 move; a full circle sweeps 2 pi."""
    x0, y0, _ = move["p0"]
    x1, y1, _ = move["p1"]
    cx, cy = x0 + move["ij"][0], y0 + move["ij"][1]
    a0, a1 = math.atan2(y0 - cy, x0 - cx), math.atan2(y1 - cy, x1 - cx)
    sweep = (a1 - a0) % (2 * math.pi) if move["kind"] == "G3" else (a0 - a1) % (2 * math.pi)
    if sweep < 1e-12 and abs(x1 - x0) + abs(y1 - y0) < 1e-12:
        sweep = 2 * math.pi
    return (cx, cy), math.hypot(x0 - cx, y0 - cy), sweep


def arc_end_radius_error(move):
    """|end radius - start radius| of an arc: a valid arc has 0 (mm)."""
    (cx, cy), r, _ = arc_geometry(move)
    return abs(math.hypot(move["p1"][0] - cx, move["p1"][1] - cy) - r)


def move_length(move):
    if move["kind"] in ("G2", "G3"):
        _, r, sweep = arc_geometry(move)
        return math.hypot(r * sweep, move["p1"][2] - move["p0"][2])
    return float(np.linalg.norm(np.subtract(move["p1"], move["p0"])))


def program_time_s(moves, rapid_rate=3000.0):
    """Seconds: cutting moves at their F, rapids at an assumed rapid rate (mm/min)."""
    t = 0.0
    for m in moves:
        rate = rapid_rate if m["kind"] == "G0" else m["f"]
        t += move_length(m) / rate * 60.0
    return t


def sample_xy(moves, step=0.25):
    """Points along the XY projection of the cutting moves (for geometry checks)."""
    pts = []
    for m in moves:
        if m["kind"] == "G0":
            continue
        n = max(2, int(move_length(m) / step) + 2)
        t = np.linspace(0.0, 1.0, n)
        if m["kind"] == "G1":
            a, b = np.array(m["p0"][:2]), np.array(m["p1"][:2])
            pts.extend(a + tt * (b - a) for tt in t)
        else:
            (cx, cy), r, sweep = arc_geometry(m)
            a0 = math.atan2(m["p0"][1] - cy, m["p0"][0] - cx)
            sgn = 1.0 if m["kind"] == "G3" else -1.0
            pts.extend((cx + r * math.cos(a0 + sgn * sweep * tt),
                        cy + r * math.sin(a0 + sgn * sweep * tt)) for tt in t)
    return np.array(pts)

## Step 4: the toolpath is an offset curve
The part outline is where the cutter *edge* must go; the centre must be r away from it.
Pocket (cut inside the outline): shrink a w x h rectangle by r on every side.
Profile (cut around a part): grow it by r; the corners become arcs of radius r.

In [ ]:
def offset_rect(w, h, r, inside=True):
    """Corners (x0, y0, x1, y1) of the centre path for the rectangle [0, w] x [0, h].
    Returns None if an inside offset would make a side negative (cutter too big)."""
    if inside:
        if 2 * r > min(w, h) + 1e-12:
            return None
        return (r, r, w - r, h - r)
    return (-r, -r, w + r, h + r)


def dist_to_rect_outline(pt, w, h):
    """Distance from a point to the closed boundary of the rectangle [0, w] x [0, h]."""
    x, y = pt
    dx, dy = max(-x, 0.0, x - w), max(-y, 0.0, y - h)
    if dx > 0 or dy > 0:
        return math.hypot(dx, dy)
    return min(x, w - x, y, h - y)


def profile_gcode(w, h, r, z, f):
    """G-code that goes round the OUTSIDE of the rectangle with a cutter of radius r (CCW)."""
    p = Program()
    p.rapid(0, -r, 5)
    p.cut(z=z, f=f)
    p.cut(w, -r, f=f)
    p.arc(True, w + r, 0, 0, r)
    p.cut(w + r, h)
    p.arc(True, w, h + r, -r, 0)
    p.cut(0, h + r)
    p.arc(True, -r, h, 0, -r)
    p.cut(-r, 0)
    p.arc(True, 0, -r, r, 0)
    p.rapid(z=5)
    return p.text()

## Step 5: plan a rectangular pocket
The centre must cover a (w - D) x (h - D) rectangle. Raster (zigzag) lines run along x,
spaced by at most the stepover s. Number of lines n = ceil(span / s) + 1, spaced evenly.
Layers = ceil(depth / depth_of_cut).

In [ ]:
def plan_pocket(w, h, depth, d, doc, stepover, feed, plunge_feed=100.0):
    """Plan a raster pocket. Raises ValueError if the cutter does not fit."""
    if d > min(w, h) + 1e-12:
        raise ValueError(f"cutter diameter {d:g} mm does not fit a {w:g} x {h:g} mm pocket")
    if stepover > d + 1e-12:
        raise ValueError("stepover larger than the cutter diameter leaves uncut strips")
    span_y, line_len = h - d, w - d
    n_lines = math.ceil(span_y / stepover - 1e-9) + 1
    s_act = span_y / (n_lines - 1) if n_lines > 1 else 0.0
    layers = math.ceil(depth / doc - 1e-9)
    per_layer = n_lines * line_len + span_y
    cut_len = layers * per_layer
    plunge_len = depth + 0.5          # the first plunge also covers 0.5 mm of air above the stock
    time_s = cut_len / feed * 60 + plunge_len / plunge_feed * 60
    return {"r": d / 2, "n_lines": n_lines, "stepover_actual": s_act, "layers": layers,
            "dz": depth / layers, "per_layer": per_layer, "cut_len": cut_len,
            "plunge_len": plunge_len, "time_s": time_s}


def pocket_gcode(w, h, depth, d, doc, stepover, feed, plunge_feed=100.0):
    """Raster pocket G-code. Pocket corner at (0, 0), stock top at Z = 0, floor at Z = -depth."""
    pl = plan_pocket(w, h, depth, d, doc, stepover, feed, plunge_feed)
    r, p = pl["r"], Program()
    p.rapid(z=5)
    p.rapid(r, r)
    p.rapid(z=0.5)
    for k in range(1, pl["layers"] + 1):
        if k > 1:
            p.rapid(r, r)
        p.cut(z=-k * pl["dz"], f=plunge_feed)
        for j in range(pl["n_lines"]):
            if j > 0:
                p.cut(y=r + j * pl["stepover_actual"], f=feed)
            p.cut(x=(w - r) if j % 2 == 0 else r, f=feed)
    p.rapid(z=5)
    return p.text()

## Step 6: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    w, h, r = 40.0, 30.0, 3.0
    print("inside offset  :", offset_rect(w, h, r, True))
    print("outside offset :", offset_rect(w, h, r, False))
    prof = parse_gcode(profile_gcode(w, h, r, -2, 200))
    print(f"profile path length = {sum(move_length(m) for m in prof if m['kind'] != 'G0'):.2f} mm")
    pl = plan_pocket(40, 30, 5, 6, 2, 4.5, 300)
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in pl.items()})
    prog = pocket_gcode(40, 30, 5, 6, 2, 4.5, 300)
    print(prog.splitlines()[:12])
    print(f"parsed time = {program_time_s(parse_gcode(prog)):.1f} s")

    fig, axs = plt.subplots(1, 2, figsize=(10, 4.6))
    ax = axs[0]
    ax.add_patch(plt.Rectangle((0, 0), w, h, fill=False, ec="k", lw=2, label="part outline"))
    ax.plot(*sample_xy(prof).T, "C0-", lw=1.5, label="outside centre path (lines + arcs)")
    x0, y0, x1, y1 = offset_rect(w, h, r, True)
    ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], "C3--", lw=1.5, label="inside centre path (pocket)")
    ax.set_aspect("equal")
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
    ax.set_title("Toolpath = outline offset by the cutter radius (3 mm)")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(alpha=0.3)
    ax = axs[1]
    first = [m for m in parse_gcode(prog) if m["kind"] == "G1" and m["p0"][2] == m["p1"][2] == -pl["dz"]]
    ax.add_patch(plt.Rectangle((0, 0), 40, 30, fill=False, ec="k", lw=2))
    ax.plot(*sample_xy(first).T, "C0-", lw=1)
    ax.set_aspect("equal")
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
    ax.set_title("First layer of the 40 x 30 mm pocket: 7 lines")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "offsets_and_pocket.png", dpi=120)

    ss = np.linspace(1.0, 6.0, 51)
    ts = [plan_pocket(40, 30, 5, 6, 2, s, 300)["time_s"] for s in ss]
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(ss, ts)
    ax2.set_xlabel("stepover (mm)")
    ax2.set_ylabel("machining time (s)")
    ax2.set_title("Pocket time against stepover (F = 300 mm/min)")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "time_vs_stepover.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the stepover in `plan_pocket` to 3 mm and predict the number of lines (9) before you run it.
2. **Stretch.** Add a finishing lap round the pocket wall to `pocket_gcode`, parse the program back and compare its length and time with the plan.
3. **Explain.** Explain why `n = ceil(span / s) + 1` has a plus one, and what would go wrong with `ceil(span / s)` alone.

## Assessment
- **A. Conceptual.** The same start (10, 0) and end (0, 10) with I-10 J0 are used for G3 and for G2. Do they give the same arc? Why does the centre not change?
- **B. Numerical.** An arc starts at (20, 0), ends at (0, 20) and is written G3 X0 Y20 I-20 J0. Find the centre, check the end radius and find the arc length.
- **C. Graph and simulation interpretation.** On the time-against-stepover graph, the time is 282.9 s at a stepover of 2 mm and 119.7 s at 6 mm. Why does the curve fall quickly at first and then flatten? Why can the stepover not exceed 6 mm?
- **D. Design and optimisation challenge.** Design: repeat the plan with an 8 mm cutter (stepover at most 75 percent of D, depth of cut 2 mm, 300 mm/min, plunge 100 mm/min). How many lines, how much cutting length and what time? Is it faster than the 6 mm cutter, and what is the catch?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    p = L.Program()
    p.cut(10, 5, f=300)
    checks.append(("G-code word formatting: the line is exactly 'G1 X10 Y5 F300'", p.lines[-1] == "G1 X10 Y5 F300"))
    checks.append(("number formatting drops trailing zeros and negative zero",
                   L.num(3.0) == "3" and L.num(-0.0) == "0" and L.num(1.23456) == "1.235"))
    mv = L.parse_gcode("G21\nG90\nG1 X10 F200 (cut)\nY5 ; same feed, still G1\nG0 Z5")
    checks.append(("parser is modal: Y5 keeps G1, F200 and X10",
                   len(mv) == 3 and mv[1]["kind"] == "G1" and mv[1]["f"] == 200 and mv[1]["p1"] == (10.0, 5.0, 0.0)))
    checks.append(("parser reads a rapid G0 Z5 as a rapid move", mv[2]["kind"] == "G0" and mv[2]["p1"][2] == 5.0))
    checks.append(("round trip: write 'G1 X12.5 Y-3 F150', parse it, get the same numbers",
                   L.parse_gcode("G1 X12.5 Y-3 F150")[0]["p1"] == (12.5, -3.0, 0.0)))
    # offsets
    inn, out = L.offset_rect(40, 30, 3, True), L.offset_rect(40, 30, 3, False)
    checks.append(("worked example: inside offset of a 40 x 30 rectangle by 3 mm is 34 x 24",
                   (inn[2] - inn[0], inn[3] - inn[1]) == (34, 24)))
    checks.append(("outside offset of a 40 x 30 rectangle by 3 mm is 46 x 36",
                   (out[2] - out[0], out[3] - out[1]) == (46, 36)))
    checks.append(("edge: a 31 mm cutter (r 15.5) cannot offset a 30 mm wide pocket inwards",
                   L.offset_rect(40, 30, 15.5, True) is None and L.offset_rect(40, 30, 15, True) is not None))
    prof = L.parse_gcode(L.profile_gcode(40, 30, 3, -2, 200))
    cutting = [m for m in prof if m["kind"] != "G0" and m["p0"][2] == m["p1"][2]]
    pts = L.sample_xy(cutting)
    dev = max(abs(L.dist_to_rect_outline(q, 40, 30) - 3.0) for q in pts)
    checks.append(("every point of the profile toolpath (lines and arcs) is 3 mm from the part outline", dev < 1e-9))
    prof_len = sum(L.move_length(m) for m in cutting)
    checks.append(("profile length = 2 (40 + 30) + 2 pi 3 = 158.85 mm", abs(prof_len - (140 + 2 * math.pi * 3)) < 1e-6))
    # arcs
    ccw = L.parse_gcode("G0 X10 Y0\nG3 X0 Y10 I-10 J0 F200")[1]
    (c, r, sw) = L.arc_geometry(ccw)
    checks.append(("G3 arc (10,0) to (0,10) with I-10 J0: centre (0,0), radius 10", c == (0.0, 0.0) and abs(r - 10) < 1e-12))
    checks.append(("that arc ends on the circle (radius error 0) and sweeps 90 degrees",
                   L.arc_end_radius_error(ccw) < 1e-12 and abs(sw - math.pi / 2) < 1e-12))
    checks.append(("its length is 10 * pi/2 = 15.708 mm", abs(L.move_length(ccw) - 10 * math.pi / 2) < 1e-9))
    cw = L.parse_gcode("G0 X10 Y0\nG2 X0 Y10 I-10 J0 F200")[1]
    checks.append(("the same endpoints with G2 (clockwise) sweep 270 degrees, length 47.124 mm",
                   abs(L.arc_geometry(cw)[2] - 1.5 * math.pi) < 1e-12 and abs(L.move_length(cw) - 15 * math.pi) < 1e-9))
    full = L.parse_gcode("G0 X10 Y0\nG3 X10 Y0 I-10 J0 F200")[1]
    checks.append(("edge: end = start with I, J is a full circle, length 62.832 mm", abs(L.move_length(full) - 20 * math.pi) < 1e-9))
    bad = L.parse_gcode("G0 X10 Y0\nG3 X0 Y9 I-10 J0 F200")[1]
    checks.append(("an inconsistent arc (end 9 mm from the centre, start 10 mm) is detected: error 1 mm",
                   abs(L.arc_end_radius_error(bad) - 1.0) < 1e-12))
    # pocket plan
    pl = L.plan_pocket(40, 30, 5, 6, 2, 4.5, 300)
    checks.append(("worked example: 4.5 mm stepover on a 24 mm span needs 7 lines, spaced 4 mm",
                   pl["n_lines"] == 7 and abs(pl["stepover_actual"] - 4.0) < 1e-12))
    checks.append(("5 mm deep at 2 mm per pass needs 3 layers of 1.667 mm",
                   pl["layers"] == 3 and abs(pl["dz"] - 5 / 3) < 1e-12))
    checks.append(("hand calculation: one layer = 7 * 34 + 24 = 262 mm, three layers 786 mm",
                   pl["per_layer"] == 262 and pl["cut_len"] == 786))
    checks.append(("worked example: time = 786/300 min + 5.5/100 min = 160.5 s", abs(pl["time_s"] - 160.5) < 1e-9))
    prog = L.pocket_gcode(40, 30, 5, 6, 2, 4.5, 300)
    moves = L.parse_gcode(prog)
    nonrapid = sum(L.move_length(m) / m["f"] * 60 for m in moves if m["kind"] != "G0")
    checks.append(("parsing our own G-code gives the planned time (cutting + plunging) 160.5 s", abs(nonrapid - pl["time_s"]) < 1e-6))
    checks.append(("adding rapids at 3000 mm/min makes the total 162.6 s", abs(L.program_time_s(moves) - 162.64) < 0.05))
    cut = [m for m in moves if m["kind"] == "G1" and m["p0"][2] == m["p1"][2]]
    xy = L.sample_xy(cut)
    checks.append(("all cutting points stay inside the centre rectangle 3..37 x 3..27 (cutter edge reaches 0..40 x 0..30)",
                   xy[:, 0].min() >= 3 - 1e-9 and xy[:, 0].max() <= 37 + 1e-9 and xy[:, 1].min() >= 3 - 1e-9 and xy[:, 1].max() <= 27 + 1e-9))
    checks.append(("the deepest Z in the program is -5 mm", abs(min(m["p1"][2] for m in moves) + 5.0) < 1e-9))
    ys = sorted({round(m["p1"][1], 6) for m in cut})
    checks.append(("the line spacing is 4 mm: Y = 3, 7, 11, 15, 19, 23, 27", ys == [3, 7, 11, 15, 19, 23, 27]))
    # limits and trade-off
    checks.append(("edge: stepover 4 on a 24 mm span gives 7 lines, not 8 (round-off safe)", L.plan_pocket(40, 30, 5, 6, 2, 4, 300)["n_lines"] == 7))
    checks.append(("a smaller stepover costs time: 2 mm gives 13 lines and 282.9 s, 6 mm gives 5 lines and 119.7 s",
                   L.plan_pocket(40, 30, 5, 6, 2, 2, 300)["n_lines"] == 13
                   and abs(L.plan_pocket(40, 30, 5, 6, 2, 2, 300)["time_s"] - 282.9) < 1e-6
                   and abs(L.plan_pocket(40, 30, 5, 6, 2, 6, 300)["time_s"] - 119.7) < 1e-6))
    failed = []
    for args in ((40, 30, 5, 31, 2, 4, 300), (40, 30, 5, 6, 2, 7, 300)):
        try:
            L.plan_pocket(*args)
            failed.append(False)
        except ValueError:
            failed.append(True)
    checks.append(("the planner refuses a cutter wider than the pocket and a stepover above the diameter", all(failed)))
    checks.append(("doubling the feed halves the cutting time", abs(L.plan_pocket(40, 30, 5, 6, 2, 4.5, 600)["cut_len"] / 600 - pl["cut_len"] / 300 / 2) < 1e-12))
    checks.append(("worked example: cutting time 786 / 300 min = 2.62 min = 157.2 s and plunge 5.5 mm / 100 mm/min = 3.3 s",
                   abs(pl["cut_len"] / 300 - 2.62) < 1e-12 and abs(pl["cut_len"] / 300 * 60 - 157.2) < 1e-9
                   and pl["plunge_len"] == 5.5 and abs(pl["plunge_len"] / 100 * 60 - 3.3) < 1e-9))
    checks.append(("snippet: 7 lines, 262 mm per layer, 52.4 s per layer", pl["n_lines"] == 7 and pl["per_layer"] == 262 and abs(262 / 300 * 60 - 52.4) < 1e-9))
    p8 = L.plan_pocket(40, 30, 5, 8, 2, 6, 300)
    checks.append(("design answer: 8 mm cutter, stepover 6 mm: 5 lines 5.5 mm apart, 182 mm per layer, 546 mm, 109.2 s cutting, 112.5 s total",
                   p8["n_lines"] == 5 and abs(p8["stepover_actual"] - 5.5) < 1e-12 and p8["per_layer"] == 182 and p8["cut_len"] == 546
                   and abs(p8["cut_len"] / 300 * 60 - 109.2) < 1e-9 and abs(p8["time_s"] - 112.5) < 1e-9))
    checks.append(("exercise: 3 mm stepover gives 9 lines and 6 mm stepover gives 5 lines",
                   L.plan_pocket(40, 30, 5, 6, 2, 3, 300)["n_lines"] == 9 and L.plan_pocket(40, 30, 5, 6, 2, 6, 300)["n_lines"] == 5))
    a20 = L.parse_gcode("G0 X20 Y0\nG3 X0 Y20 I-20 J0 F200")[1]
    checks.append(("assessment B: arc (20,0) to (0,20), I-20 J0: centre (0,0), end radius error 0, length 31.416 mm",
                   L.arc_geometry(a20)[0] == (0.0, 0.0) and L.arc_end_radius_error(a20) < 1e-12 and abs(L.move_length(a20) - 10 * math.pi) < 1e-9))
    q = L.parse_gcode("G0 X10 Y0\nG3 X0 Y10 I-10 J0 F100")[1]
    checks.append(("roadmap: the quarter circle of radius 10 mm has length 15.708 mm", abs(L.move_length(q) - 15.708) < 1e-3))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
